# Ch 4 - Implement GPT-2

In [6]:
import numpy as np
import tiktoken
import torch
import torch.nn as nn

from gpt_download import download_and_load_gpt2
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Model

2025-04-02 10:18:01.546632: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/Users/andylee/anaconda3/envs/llm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Model Architecture
This section defines the model architecture to instantiate the NN model.

In [8]:
class MultiHeadAttention(nn.Module):
    """Implementation of multihead attention w/ parallel matrix processing"""

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        # validate input dimensions
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"
        self.d_out = d_out
        self.num_heads = num_heads
        self.attention_dim = d_out // num_heads

        # setup attention matrices
        self.W_query = nn.Linear(in_features=d_in, out_features=d_out, bias=qkv_bias)
        self.W_key = nn.Linear(in_features=d_in, out_features=d_out, bias=qkv_bias)
        self.W_value = nn.Linear(in_features=d_in, out_features=d_out, bias=qkv_bias)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

        # setup dropout
        self.dropout = nn.Dropout(dropout)

        # setup linear
        self.out_proj = nn.Linear(d_out, d_out)

    
    def forward(self, x):
        n, seq_length, _ = x.shape

        # compute Q, K, V matrices
        x_query = self.W_query(x)
        x_key = self.W_key(x)
        x_value = self.W_value(x)

        # reshape to separate into Q = [Q1, Q2, ...], K = [K1, K2, ...]
        x_query = x_query.view(n, seq_length, self.num_heads, self.attention_dim)
        x_query = x_query.transpose(1, 2) # (n, num_heads, seq_length, attention_dim)

        x_key = x_key.view(n, seq_length, self.num_heads, self.attention_dim)
        x_key = x_key.transpose(1, 2) # (n, num_heads, seq_length, attention_dim)
        x_key = x_key.transpose(2, 3) # (n, num_heads, attention_dim, seq_length)

        x_value = x_value.view(n, seq_length, self.num_heads, self.attention_dim)
        x_value = x_value.transpose(1, 2) # (n, num_heads, seq_length, attention_dim)

        # compute attention scores (per-head)
        dk_constant = x_key.shape[-1] ** -0.5
        mask_context = self.mask.bool()[:seq_length, :seq_length] 
        attention_scores = (x_query @ x_key) # (n, num_heads, seq_length, seq_length)
        attention_scores.masked_fill_(mask_context, -torch.inf)

        # compute attention weights 
        # note : no dropout on scores (b/c dropout on -inf is not well-defined)
        attention_weights = torch.softmax(attention_scores * dk_constant, dim=-1)
        attention_weights = self.dropout(attention_weights)

        # compute context
        context = attention_weights @ x_value # (n, num_heads, seq_length, attn_dim)
        context = context.transpose(1, 2) # (n, seq_length, num_heads, attn_dim) , done by convention
        context = context.contiguous().view(n, seq_length, self.d_out) # (n, seq_length, d_out)
        
        # apply linear layer
        return self.out_proj(context)


In [5]:
class GPTModel(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, x):

        # read input X
        _, seq_len = x.shape

        # map X to embedding space
        # embedding matrices take list of indices
        tok_embeds = self.tok_emb(x)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=x.device))
        x = tok_embeds + pos_embeds

        # process embed(X) through architecture
        # note : 12 transformer blocks is main processing
        x = self.drop_emb(x)
        x = self.trf_blocks(x) 
        x = self.final_norm(x)

        logits = self.out_head(x)
        return logits

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift
    
class GELU(nn.Module):
    """Activation function that's smoother than RELU"""
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))
        ))
    
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
)
    def forward(self, x):
        return self.layers(x)
    
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x


## GPT-2 Factory
Load pretrained GPT-2 from HF and OpenAI

In [13]:
def get_model_config(model_name):
    base_config = {
        "vocab_size": 50257,     # Vocabulary size
        "context_length": 1024,  # Context length
        "drop_rate": 0.1, # Dropout rate
        "qkv_bias": True# Query-Key-Value bias
    }

    model_configs = {
        "gpt2-small": {"emb_dim": 768, "n_layers": 12, "n_heads": 12}, #124m
        "gpt2-medium": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16}, #355m
        "gpt2-large": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20}, #774m
        "gpt2-xl": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25}, #1558m
    }

    base_config.update(model_configs[model_name])
    base_config.update({'model_name': model_name})
    return base_config


def assign_check(left, right):
    """Utility function to ensure loading is correct"""
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

def load_pretrained_weights_openai(gpt, params):
    """Load pretrained weights from OpenAI
    :param gpt: GPTModel
    :param params: dict (pretrained weights from openAI)
    """
    gpt.pos_emb.weight = assign_check(gpt.pos_emb.weight, params['wpe'])
    gpt.tok_emb.weight = assign_check(gpt.tok_emb.weight, params['wte'])
    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split((params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign_check(gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign_check(gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign_check(gpt.trf_blocks[b].att.W_value.weight, v_w.T)
        q_b, k_b, v_b = np.split((params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign_check(gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign_check(gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign_check(gpt.trf_blocks[b].att.W_value.bias, v_b)
        gpt.trf_blocks[b].att.out_proj.weight = assign_check(gpt.trf_blocks[b].att.out_proj.weight, params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign_check(gpt.trf_blocks[b].att.out_proj.bias, params["blocks"][b]["attn"]["c_proj"]["b"])
        gpt.trf_blocks[b].ff.layers[0].weight = assign_check(gpt.trf_blocks[b].ff.layers[0].weight, params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign_check(gpt.trf_blocks[b].ff.layers[0].bias, params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign_check(gpt.trf_blocks[b].ff.layers[2].weight, params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign_check(gpt.trf_blocks[b].ff.layers[2].bias, params["blocks"][b]["mlp"]["c_proj"]["b"])
        gpt.trf_blocks[b].norm1.scale = assign_check(gpt.trf_blocks[b].norm1.scale, params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign_check(gpt.trf_blocks[b].norm1.shift, params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign_check(gpt.trf_blocks[b].norm2.scale, params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign_check(gpt.trf_blocks[b].norm2.shift, params["blocks"][b]["ln_2"]["b"])
    gpt.final_norm.scale = assign_check(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign_check(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign_check(gpt.out_head.weight, params["wte"])

def get_pretrained_gpt(model_name="gpt2-small", model_provider="openai"):
    # load model architecture
    config = get_model_config(model_name)
    gpt_model = GPTModel(config)
    gpt_model.eval()

    # get pretrained params
    _, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

    # load weights
    if model_provider == "hf":
        raise NotImplementedError("HF model loading not implemented")

    elif model_provider == "openai":
        load_pretrained_weights_openai(gpt_model, params)

    return gpt_model


# script to download model weights (note : 500mb download)
# settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

In [14]:
gpt_model = get_pretrained_gpt(model_name="gpt2-small", model_provider="openai")

File already exists and is up-to-date: gpt2/124M/checkpoint
File already exists and is up-to-date: gpt2/124M/encoder.json
File already exists and is up-to-date: gpt2/124M/hparams.json
File already exists and is up-to-date: gpt2/124M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/124M/model.ckpt.index
File already exists and is up-to-date: gpt2/124M/model.ckpt.meta
File already exists and is up-to-date: gpt2/124M/vocab.bpe


AttributeError: 'NoneType' object has no attribute 'shape'

# Run Model 

In [24]:
def generate_text_simple(model, x, max_new_tokens=100, context_size=10):

    # iteratively generate new tokens (up to a limit)
    for _ in range(max_new_tokens):

        # get context for prediction
        # note: equivalent to x[:, seq_length - context_size:]
        x_conditioned = x[:, -context_size:] 

        # run model to predict next token
        with torch.no_grad():
            logits = model(x_conditioned)

        # decode next token prediction
        next_token_logits = logits[:, -1, :]
        next_token_probabilities = torch.softmax(next_token_logits, dim=-1)
        next_token_prediction = torch.argmax(next_token_probabilities, dim=-1, keepdim=True)
        # next_token_prediction = torch.multinomial(next_token_probabilities, dim=-1, keepdim=True)

        # create next context 
        x = torch.cat((x_conditioned, next_token_prediction), dim=1)

    return x


In [9]:
# set seed
torch.manual_seed(123)

# setup tokenizer
tokenizer = tiktoken.get_encoding("gpt2")

# setup data
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0) # (n x seq_length), each element is list of indices [idx1, idx2, ...]
print(f"batch shape: {batch.shape}")

# instantiate model
model = GPTModel(GPT_CONFIG_124M)
gpt2_large_model = GPTModel(GPT2_LARGE_CONFIG)
logits = model(batch)

# inspect output
print(f"logits_shape : {logits.shape}")

# inspect parameters
# exercise 4.1
model = GPTModel(GPT2_LARGE_CONFIG)
total_params_full = sum(p.numel() for p in model.parameters())
total_params_out_head = sum(p.numel() for p in model.out_head.parameters())
total_params_transformer_block = sum(p.numel() for p in model.trf_blocks.parameters())

trf_block = model.trf_blocks[0]
trf_attn = trf_block.att
trf_ff = trf_block.ff
total_params_single_attn = sum(p.numel() for p in trf_attn.parameters())
total_params_single_ff = sum(p.numel() for p in trf_ff.parameters())

# computer parameters
print(f"Total parameters in the full model: {total_params_full}")
print(f"Total parameters in the output head: {total_params_out_head}")
print(f"Total parameters in the transformer block: {total_params_transformer_block}")
print(f"Total parameters in a single attention layer: {total_params_single_attn}")
print(f"Total parameters in a single feed-forward layer: {total_params_single_ff}")

# compute memory requirements
total_size_bytes = total_params_full * 4
total_size_mb = total_size_bytes / (1024 * 1024)
print(f"Total size of the model: {total_size_mb:.2f} MB")

batch shape: torch.Size([2, 4])
logits_shape : torch.Size([2, 4, 50257])
Total parameters in the full model: 838220800
Total parameters in the output head: 64328960
Total parameters in the transformer block: 708249600
Total parameters in a single attention layer: 6554880
Total parameters in a single feed-forward layer: 13113600
Total size of the model: 3197.56 MB


# Ch. 5 - Pretraining

In [29]:

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())

# initialize start context
start_context = "Hello, I am"
start_context = "Every effort moves you"

# setup
tokenizer = tiktoken.get_encoding("gpt2")
model = GPTModel(GPT_CONFIG_124M)
model.eval()

# prediction
token_ids = generate_text_simple(model=model,
                                 x=text_to_token_ids(start_context, tokenizer),
                                 max_new_tokens=10,
                                 context_size=GPT_CONFIG_124M["context_length"])
output_text = token_ids_to_text(token_ids, tokenizer)
print(f"output_text : {output_text}")


output_text : Every effort moves youIO Sukotion fascinationsinbd wakes Chero rob Exactly


## Small training loop

In [27]:
def calc_loss_batch(input_batch, target_batch, model, device):
    """Calculates loss for a single batch given input & target"""
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(
        logits.flatten(0, 1), target_batch.flatten()
    )
    return loss

def calc_loss_loader(data_loader, model, device, num_batches=None):
    """Iterates through data loader and calculates loss per batch"""
    total_loss = 0.

    # setup
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    # iterate through batches , compute losses
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break

    return total_loss / num_batches


In [11]:
# Example : 1 training loops

# input data
file_path = "/Users/andylee/Projects/llm-from-scratch/data/the-verdict.txt"
# file_path = "/datasets/the-verdict/the-verdict.txt"
with open(file_path, "r", encoding="utf-8") as file:
    text_data = file.read()

# split to train & validation batch
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

# create data loaders
train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)
val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

# setup devices
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 
model.to(device)

# run 1 single iteration of loss
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)
print("Training loss:", train_loss)
print("Validation loss:", val_loss)


Training loss: 11.019258499145508
Validation loss: 11.028642654418945


In [26]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(
            train_loader, model, device, num_batches=eval_iter
        )
        val_loss = calc_loss_loader(
            val_loader, model, device, num_batches=eval_iter
        )
    model.train()
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context):
    """Run a single forward pass of model to get predicted tokens"""

    # set to eval mode
    model.eval()

    # encode X 
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    context_size = model.pos_emb.weight.shape[0]

    # predict y
    with torch.no_grad():
        token_ids = generate_text_simple(model=model, x=encoded, max_new_tokens=50, context_size=context_size)

    # get text(y)
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text)

    # revert to train mode
    model.train()


def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs, eval_freq, eval_iter, start_context, tokenizer):
    """Training loop of NN"""
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()

        # iterate through training batches 
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()

            tokens_seen += input_batch.numel()
            global_step += 1

            # print losses based on eval_freq
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # generate new token predictions after each epoch 
        generate_and_print_sample(model, tokenizer, device, start_context)
    
    return train_losses, val_losses, track_tokens_seen


In [27]:
torch.manual_seed(123)

# make the GPT model
model = GPTModel(GPT_CONFIG_124M)
model.to(device)

# setup training optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0004, weight_decay=0.1
)

# start training!! 
num_epochs = 2
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Every effort should move you forward", tokenizer=tokenizer
)


Ep 1 (Step 000000): Train loss 9.785, Val loss 9.926
Ep 1 (Step 000005): Train loss 7.987, Val loss 8.335
Every effort should move you forward,,,,,,,,,,.







































Ep 2 (Step 000010): Train loss 6.755, Val loss 7.048
Ep 2 (Step 000015): Train loss 6.118, Val loss 6.576
Every effort should move you forward,, and,, and,,,, and, and,,,,,,,,,,,,,, and,,,, and,,,,,,,, and,,,,,, and,


In [14]:
# Utility functions

def assign_check(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

def generate(model, idx, max_new_tokens, context_size,
             temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(
                logits < min_val,
                torch.tensor(float('-inf')).to(logits.device),
                logits
            )
        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        if idx_next == eos_id:
            break
        idx = torch.cat((idx, idx_next), dim=1)
    return idx


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

oai_gpt = load_open_ai()
hf_gpt = load_hf()

oai_gpt.to(device)
hf_gpt.to(device)

File already exists and is up-to-date: gpt2/124M/checkpoint
File already exists and is up-to-date: gpt2/124M/encoder.json
File already exists and is up-to-date: gpt2/124M/hparams.json
File already exists and is up-to-date: gpt2/124M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/124M/model.ckpt.index
File already exists and is up-to-date: gpt2/124M/model.ckpt.meta
File already exists and is up-to-date: gpt2/124M/vocab.bpe


/var/folders/k8/kfnfm34s37gb64xyqjkjztz40000gn/T/ipykernel_85672/4200491702.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.nn.Parameter(torch.tensor(right))


Output text:
 Every effort moves you forward now as you get more money out if a large portion go straight into those reserves... but after so many years your plan of


In [37]:
token_ids_1 = generate(model=oai_gpt,
                     idx=text_to_token_ids("Every effort moves you forward", tokenizer).to(device), 
                     max_new_tokens=25,
                     context_size=1024,
                     top_k=50,
                     temperature=1.5)

token_ids_2 = generate(model=hf_gpt,
                     idx=text_to_token_ids("Every effort moves you forward", tokenizer).to(device), 
                     max_new_tokens=25,
                     context_size=1024,
                     top_k=50,
                     temperature=1.5)


print("Output text (openai):\n", token_ids_to_text(token_ids_1, tokenizer))
print("Output text (huggingface):\n", token_ids_to_text(token_ids_2, tokenizer))

Output text (openai):
 Every effort moves you forward will need before you've seen to succeed was put in to help her gain ground; that much is evident in knowing she is
Output text (huggingface):
 Every effort moves you forward towards making the case your entire life is so slowly approaching and quickly. Just as a car will run about like that it will


## Scratchpad